<a href="https://colab.research.google.com/github/aamanlamba/DecisionStream_Immutable_Audit_Trail_Colab_Exercise/blob/main/FDE_Immutable_Audit_Trail_Decision_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FDE Decision Lab: The Audit Trail That Cannot Be Changed

**Sprint 2 · Day 33 — Immutable Audit Trail**

### Purpose
This is an FDE-style scenario lab, not a quiz. You will make architecture and operational decisions, explain **why** you chose them, and then use an LLM to challenge your thinking.

The scenario is based on the lesson material:
- Your application code is the **recorder**.
- Cosmos DB Change Feed is the **delivery mechanism**, not the original recorder.
- Audit records are **append-only** and should be written before the case moves on.
- Cosmos is the fast working copy; Blob WORM is the evidence store.
- Immutable retention decisions must be made carefully before locking.
- PII should be kept out of immutable storage where possible.
- Azure Activity Log answers a different question: infrastructure/control-plane changes.
- An archive must be designed around the audit question, such as **“show me this case.”**

> **FDE rule:** Do not select an option because it sounds technically impressive. Select it because you can defend the decision against failure, audit, compliance, operational, and customer consequences.


## Scenario: Claims Review Platform

You are the **Forward Deployed Engineer** supporting a claims-processing platform used by an insurance customer.

A claim moves through this lifecycle:

`created → extracted → scored → routed → reviewed → complete`

The customer has an important requirement:

> **“Two years from now, show me exactly what happened to Claim CLM-2026-4471, who or what caused each transition, which model/policy was used, and prove that the historical record was not changed.”**

Current architecture:

```text
Claims API → Cosmos DB (claims)
                    │
                    └→ Change Feed → Audit Function → Blob Storage
```

A previous team assumed the Change Feed would capture every state change. Testing showed otherwise: rapid intermediate changes can be missed because the normal feed can expose the latest state rather than every intermediate version.

Your job is to redesign the audit approach and defend each decision.

### Constraints

- The application must continue processing claims even if the downstream audit/archive Function is temporarily unavailable.
- Auditors need a reliable historical trail.
- The customer may later request deletion of personal data.
- The audit archive must be difficult or impossible to tamper with.
- Operations must be able to find one claim quickly.
- You must be able to demonstrate evidence, not just say “it is immutable.”


## How to do the exercise

For every decision:

1. Choose **one option**.
2. Write your reasoning in your own words.
3. State **what could go wrong** if your choice is wrong.
4. Think like an FDE: identify assumptions, evidence you would collect, and who needs to approve the decision.
5. Only after completing the exercise, run the **AI Evaluation** section.

The evaluator will not simply mark the letter. It will assess whether your reasoning is defensible and whether you noticed important trade-offs.


---
## Q1 — Who is the recorder?

**Situation:** The team says: “Let's use Cosmos DB Change Feed as the audit recorder. It already sees changes, so we don't need extra writes.”

**Choose ONE:**

1. A. Use Change Feed as the authoritative audit recorder.
2. B. Make the application Function write a new audit row at the moment the transition happens; use Change Feed only to carry those audit rows downstream.
3. C. Ask Azure Activity Log to capture the case transition automatically.
4. D. Write one daily snapshot of every claim and call that the audit trail.

### Your decision
- **Selected option (1–4):** 2. B. Make the application Function write a new audit row at the moment the transition happens; use Change Feed only to carry those audit rows downstream.
- **Why did you choose it?** This ensures the audit record is captured independent of any changes to the Cosmos DB and if the Change Feed is unavailable, it will be still available as a reference. Additionally, retention and WORM policies can be applied to the audit log.
- **What could go wrong if your choice is wrong?** Risk of audit issues and loss of evidence.
- **What evidence would you ask the customer/system team to provide?** the policy history, proof that no changes were made to the audit log post-writing.
- **Who should approve or own the decision, if applicable?** The Head of Claims

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests whether the trainee understands the central recorder-vs-delivery distinction.


---
## Q2 — What exactly should one audit record contain?

**Situation:** A reviewer asks: “Why was this claim routed to priority review at 09:14?” Which record design is strongest?

**Choose ONE:**

1. A. `{case_id, current_state}` only.
2. B. `{case_id, to_state}` only.
3. C. `{audit_id, case_id, event, from_state, to_state, timestamp, actor, score/reasons where relevant, model_version, prompt_version, correlation_id}`.
4. D. Store only the complete claim JSON every time the state changes.

### Your decision
- **Selected option (1–4):** 3. C. `{audit_id, case_id, event, from_state, to_state, timestamp, actor, score/reasons where relevant, model_version, prompt_version, correlation_id}`.
- **Why did you choose it?** This provides all the information needed to reconstruct the record and evidence of changes of state, the actors and versions of models, prompts.
- **What could go wrong if your choice is wrong?** Excessive storage, missing evidence
- **What evidence would you ask the customer/system team to provide?** the information needed for the audit record
- **Who should approve or own the decision, if applicable?**
Head of Claims
> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests audit-record design and the ability to think about future investigation questions.


---
## Q3 — When should the audit row be written?

**Situation:** The claim is about to move from `scored` to `priority_review`. Which sequence gives the strongest audit guarantee?

**Choose ONE:**

1. A. Move the claim first, then write the audit record later.
2. B. Let an external monitoring service notice the change and create the audit record.
3. C. Write the audit record in the same application code path before the result is released / the case moves on.
4. D. Write it once per hour in a batch.

### Your decision
- **Selected option (1–4):** 3. C. Write the audit record in the same application code path before the result is released / the case moves on.
- **Why did you choose it?** Ensures Consistency and transaction accuracy. ALigns with the State change.
- **What could go wrong if your choice is wrong?** Risk of loss, audit issues
- **What evidence would you ask the customer/system team to provide?** Testing logs
- **Who should approve or own the decision, if applicable?** NA

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests ordering, race-condition thinking, and evidence timing.


---
## Q4 — What should happen if the audit write is retried?

**Situation:** The application writes an audit record successfully, but the request times out before the caller knows it succeeded. The operation is retried.

**Choose ONE:**

1. A. Generate a random audit ID each time so every attempt is visible.
2. B. Use a deterministic audit ID for the same event so a retry does not create a second logical audit row.
3. C. Ignore retries because audit writes should never fail.
4. D. Delete the first record if a duplicate is detected.

### Your decision
- **Selected option (1–4):** B. Use a deterministic audit ID for the same event so a retry does not create a second logical audit row.
- **Why did you choose it?** This serves as an idemopotency key so When the caller retries the timed-out request, the system recognizes the deterministic ID, realizes the work has already been completed successfully, and safely returns a success response without creating a duplicate, misleading log entry.
- **What could go wrong if your choice is wrong?** If the choice was wrong, we would end up with duplicate audit entries for the same event.
- **What evidence would you ask the customer/system team to provide?** NA
- **Who should approve or own the decision, if applicable?** Technology leader

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests idempotency and failure-mode thinking.


---
## Q5 — The audit Function is down for two hours

**Situation:** Claims continue processing while the downstream Audit Function is unavailable. What is the correct interpretation?

**Choose ONE:**

1. A. The audit trail is definitely lost because the Audit Function was down.
2. B. Nothing can be known until the Function restarts.
3. C. The in-line application audit records should still exist; the downstream Function should resume from its Change Feed position and carry the backlog, provided the required changes are still available.
4. D. Azure Activity Log will reconstruct the missing case transitions.

### Your decision
- **Selected option (1–4):** 3. C. The in-line application audit records should still exist; the downstream Function should resume from its Change Feed position and carry the backlog, provided the required changes are still available.
- **Why did you choose it?** f a downstream Audit Function goes down, the core application continues to process claims and write changes to the primary database. The Change Feed inherently preserves the sequential order of these records. Once the downstream Function restarts, it will read from its last checkpoint/pointer and process the backlog. The audit trail is not lost as long as the data retention period (time-to-live) of the source change feed accommodates the two-hour downtime.
- **What could go wrong if your choice is wrong?** If Option C is incorrect due to systemic misconfiguration, the following could happen:Data Loss via Retention Expiration: If the Change Feed or event log has a retention window shorter than two hours, older unprocessed messages will expire and be permanently lost before the function recovers.Lack of Local Backlogs: If the upstream application relies on synchronous direct calls to the Audit Function rather than an asynchronous Change Feed pattern, claims might process without generating any audit records at all, making Option A true.Checkpoint Corruption: If the Function’s pointer or checkpoint mechanism fails upon crash, it might skip records or process duplicates, causing data gaps or integrity issues.
- **What evidence would you ask the customer/system team to provide?** To verify this interpretation is correct, you should request:Architecture Diagram: Confirmation that an asynchronous pattern (Change Feed, Service Bus, or Event Hub) sits between the claims processor and the Audit Function.Retention Policy Settings: The configured data retention period or Time-to-Live (TTL) for the source database's Change Feed or messaging queue to ensure it exceeds two hours.Application Logs: Telemetry showing that the upstream claims processor is successfully committing transactions without throwing exceptions related to the downstream failure.
- **Who should approve or own the decision, if applicable?** The Lead Cloud/Solutions Architect and the Information Security Officer (ISO) should jointly own this validation. The Architect confirms the technical recovery mechanism, while the ISO signs off on whether the temporary state violation poses a compliance or data integrity risk during the catch-up period.

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests resilience thinking and separation of source-of-truth from downstream processing.


---
## Q6 — The outage lasts two weeks

**Situation:** The downstream processor has been broken for two weeks and relevant Change Feed history is no longer available. What should an FDE recommend?

**Choose ONE:**

1. A. Assume the Change Feed will always contain everything needed.
2. B. Restart the Function and report success without reconciliation.
3. C. Detect and alert on processing lag/retention risk, investigate the gap, and use the authoritative in-line audit records or another approved recovery source to reconcile what is missing from downstream storage.
4. D. Increase the retention period only after the incident has happened; that will recover old events.

### Your decision
- **Selected option (1–4):** 3. C. Detect and alert on processing lag/retention risk, investigate the gap, and use the authoritative in-line audit records or another approved recovery source to reconcile what is missing from downstream storage.
- **Why did you choose it?** Ensures n o loss of events and addresses processing lag.
- **What could go wrong if your choice is wrong?** If this is wrongly configured, there would be significant loss of audit events and risk at lost transactions, changes that were unauthorized, etc.
- **What evidence would you ask the customer/system team to provide?** Details on outage, logs, evidence that the Function was not modified, and retention policies
- **Who should approve or own the decision, if applicable?** Solution Architect

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests operational monitoring, retention-risk thinking, and recovery planning.


---
## Q7 — What belongs in immutable storage?

**Situation:** The team proposes storing the claimant's full name, address and bank account in the WORM audit archive because “more information is always better for audits.”

**Choose ONE:**

1. A. Accept all PII; WORM means it is automatically compliant.
2. B. Keep identifiable personal data out where possible; use case/customer references and keep deletable identifying details in a normal store.
3. C. Put every field in WORM and ask the DPO to delete it later.
4. D. Never store any audit information at all because GDPR exists.

### Your decision
- **Selected option (1–4):** 2. B. Keep identifiable personal data out where possible; use case/customer references and keep deletable identifying details in a normal store.
- **Why did you choose it?** Reduces exposure risk, ensures ability to reconstruct the record when approved by a lawyer.
- **What could go wrong if your choice is wrong?** PII exposure and leakage.
- **What evidence would you ask the customer/system team to provide?** That no information has been stored or accessed outside the secure customer PII store.
- **Who should approve or own the decision, if applicable?** Head of Privacy

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests data minimisation, compliance trade-offs, and knowing when to escalate to the data-protection/legal owner.


---
## Q8 — The WORM policy

**Situation:** You are preparing production immutable storage. Which approach is safest?

**Choose ONE:**

1. A. Lock seven years immediately so nobody can change anything.
2. B. Start with an unlocked short-retention policy for testing; agree the retention period and permitted fields with the appropriate owner; lock only after that decision is approved.
3. C. Lock one day in production and increase it later if auditors complain.
4. D. Keep the policy permanently unlocked so engineers can fix mistakes.

### Your decision
- **Selected option (1–4):** 2. B. Start with an unlocked short-retention policy for testing; agree the retention period and permitted fields with the appropriate owner; lock only after that decision is approved.
- **Why did you choose it?** Enables testing and verification of process and aligns with retention policy once approved
- **What could go wrong if your choice is wrong?** The record is locked and cannot be modified until the policy period expires.
- **What evidence would you ask the customer/system team to provide?** Retention Policy, Test results for the WORM activities while the policy is unlocked.
- **Who should approve or own the decision, if applicable?** Head of Privacy

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests irreversible-decision thinking and governance.


---
## Q9 — How do you prove immutability?

**Situation:** A project manager says: “Blob Storage is WORM, so just put that sentence in the architecture document.” What is the better FDE response?

**Choose ONE:**

1. A. Agree; documentation is enough.
2. B. Take a screenshot of the portal showing the WORM setting and stop there.
3. C. Perform evidence tests: write a record, attempt delete, attempt overwrite, attempt to shorten locked retention, read back and compare, and test search by case_id; preserve the refused-operation evidence.
4. D. Ask the cloud provider to certify it verbally.

### Your decision
- **Selected option (1–4):** 3. C. Perform evidence tests: write a record, attempt delete, attempt overwrite, attempt to shorten locked retention, read back and compare, and test search by case_id; preserve the refused-operation evidence.
- **Why did you choose it?** Ensure reliable testing and verify through evidence.
- **What could go wrong if your choice is wrong?** Risk of loss of audit records, lack of evidence to demonstrate working process for WORM.
- **What evidence would you ask the customer/system team to provide?** Test results
- **Who should approve or own the decision, if applicable?** Solution Architect

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests evidence-driven engineering rather than assertion-driven engineering.


---
## Q10 — Activity Log vs application audit

**Situation:** An auditor asks: “Who deleted the storage container?” Which source should you use, and what can it prove?

**Choose ONE:**

1. A. The application audit trail; it records all Azure infrastructure actions.
2. B. Azure Activity Log; it records Azure resource/control-plane actions. It complements the application audit trail but does not explain why a claim was flagged.
3. C. Cosmos Change Feed; it records every Azure subscription action.
4. D. Blob WORM; it automatically records who touched Azure resources.

### Your decision
- **Selected option (1–4):** 2. B. Azure Activity Log; it records Azure resource/control-plane actions. It complements the application audit trail but does not explain why a claim was flagged.
- **Why did you choose it?** Control-plane tracking: Azure Activity Logs record management operations like container or resource deletions performed via Azure Resource Manager.Identity capture: It identifies the specific user or service principal identity (caller) who initiated the deletion.Default availability: It is captured by the platform automatically without requiring prior custom application configuration.
- **What could go wrong if your choice is wrong?** Data Gaps, Retention policy violation, evidence gaps
- **What evidence would you ask the customer/system team to provide?** Activity Logs, access management policy
- **Who should approve or own the decision, if applicable?** CISO, Cloud Infra Team

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests architectural boundary thinking and avoiding tool misuse.


---
## Q11 — Archive layout

**Situation:** Two years later, an auditor asks for exactly one claim. You have about half a million audit rows in Blob Storage. Which layout best matches the question “show me this case”?

**Choose ONE:**

1. A. Put all records into large sequential batch files and scan every file.
2. B. Use a layout organised around case_id, such as `audit/case_id=<id>/<date>.json`, so the relevant case can be located directly.
3. C. Use a random folder name so access is unpredictable.
4. D. Store only a database backup because Blob search is unnecessary.

### Your decision
- **Selected option (1–4):** 2. B. Use a layout organised around case_id, such as `audit/case_id=<id>/<date>.json`, so the relevant case can be located directly.
- **Why did you choose it?** Ease of access, low cost - reading only one file out of thousands.
- **What could go wrong if your choice is wrong?** Slow search, higher costs, missed deadlines.
- **What evidence would you ask the customer/system team to provide?** Storage Layout Proof: A sample path showing how folders use the case_id.Access Test Results: Proof of how fast a single test query runs.Cost Estimates: Reports on expected read operations for audits.
- **Who should approve or own the decision, if applicable?** Data Architect, Compliance Lead

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests query-driven data layout and long-term operational thinking.


---
## Q12 — What does “audit trail complete” mean?

**Situation:** Your dashboard says the audit pipeline is green. An auditor asks: “Can you prove every important transition was captured?” What is the strongest response?

**Choose ONE:**

1. A. “The Function had no errors, so yes.”
2. B. “The Change Feed is enabled, so yes.”
3. C. “Here is what our application records, here is the evidence that downstream delivery and storage are working, here is what the trail does not capture, and here is our recovery/monitoring approach.”
4. D. “Azure Activity Log confirms the application was healthy.”

### Your decision
- **Selected option (1–4):** 3. C. “Here is what our application records, here is the evidence that downstream delivery and storage are working, here is what the trail does not capture, and here is our recovery/monitoring approach.”
- **Why did you choose it?** Completeness: It addresses the exact question of proof across the entire pipeline, not just one component.Honesty: It openly states what the audit trail does not capture, which builds trust with the auditor.Accountability: It shows you actively monitor and recover the system if something fails.Rejection of false security: Options A, B, and D assume that a healthy function, an enabled log feature, or a green status light equals verified, complete data capture, which is rarely true in a strict audit.
- **What could go wrong if your choice is wrong?** Audit Failure, Data Loss, Loss of Trust
- **What evidence would you ask the customer/system team to provide?** Trace Logs, Recon Reports, Dead-letter Queue Logs, Access and Retention metrics
- **Who should approve or own the decision, if applicable?** Compliance Officer, Architecture Lead, Security Lead.

> **Do not look at the answer key yet.** The reference reasoning is stored later in this notebook so the AI evaluator can compare your reasoning with it.

**FDE lens:** Tests the overall “Can you prove it?” mindset.


---
# AI Evaluation

You can evaluate your answers using either:

- **Gemini** — suitable for a lightweight/free-tier experimentation setup where available through Google AI Studio.
- **Azure OpenAI** — suitable when the customer wants evaluation to remain inside an Azure-controlled environment.

### Important
This notebook does **not** contain an API key. You supply your own key through an environment variable or a notebook input cell.

The evaluator:
1. Reads your selected option and reasoning.
2. Compares it with the lesson-grounded reference reasoning.
3. Asks the selected model to explain **where it agrees and differs**.
4. Scores your reasoning on:
   - Decision correctness
   - FDE reasoning
   - Failure-mode awareness
   - Evidence thinking
   - Governance/compliance awareness
5. Produces a recommended answer and explains why.


In [1]:
# Install dependencies if required.
# Run this cell once in a fresh environment.

%pip -q install requests


In [16]:
import os
import json
import textwrap
import requests
from getpass import getpass

# Choose one:
PROVIDER = "gemini"       # "gemini" or "azure"

# ---- Gemini ----
# Get a Gemini API key from Google AI Studio and paste it when prompted.
# The exact free-tier model availability can change, so set the model explicitly.
GEMINI_MODEL = "gemini-2.5-flash"

# ---- Azure OpenAI ----
# Fill these only when PROVIDER = "azure".
AZURE_ENDPOINT = ""       # e.g. https://YOUR-RESOURCE.openai.azure.com
AZURE_DEPLOYMENT = ""     # your deployed model name
AZURE_API_VERSION = "2024-10-21"

if PROVIDER == "gemini":
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Enter Gemini API key: ")
else:
    AZURE_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY") or getpass("Enter Azure OpenAI API key: ")

print("Provider selected:", PROVIDER)


Enter Gemini API key: ··········
Provider selected: gemini


In [4]:
# The lesson-grounded reference rubric.
# This is intentionally kept separate from the trainee answer cells.

QUESTIONS = [{'id': 1, 'title': 'Q1 — Who is the recorder?', 'situation': "The team says: “Let's use Cosmos DB Change Feed as the audit recorder. It already sees changes, so we don't need extra writes.”", 'options': ['A. Use Change Feed as the authoritative audit recorder.', 'B. Make the application Function write a new audit row at the moment the transition happens; use Change Feed only to carry those audit rows downstream.', 'C. Ask Azure Activity Log to capture the case transition automatically.', 'D. Write one daily snapshot of every claim and call that the audit trail.'], 'reference_reasoning': 'Best reasoning: **B**. The lesson explicitly distinguishes the recorder from the delivery mechanism. The application code should write the transition record when the event happens; Change Feed then carries those audit rows to downstream storage. This avoids losing rapid intermediate states.', 'correct_option': 2, 'fde_lens': 'Tests whether the trainee understands the central recorder-vs-delivery distinction.'}, {'id': 2, 'title': 'Q2 — What exactly should one audit record contain?', 'situation': 'A reviewer asks: “Why was this claim routed to priority review at 09:14?” Which record design is strongest?', 'options': ['A. `{case_id, current_state}` only.', 'B. `{case_id, to_state}` only.', 'C. `{audit_id, case_id, event, from_state, to_state, timestamp, actor, score/reasons where relevant, model_version, prompt_version, correlation_id}`.', 'D. Store only the complete claim JSON every time the state changes.'], 'reference_reasoning': 'Best reasoning: **C**. The transition itself is the important fact. The lesson highlights deterministic audit_id, actor, from_state/to_state, timestamp, model version, prompt version and correlation ID. The exact fields should still be agreed with the customer and data-protection owner.', 'correct_option': 3, 'fde_lens': 'Tests audit-record design and the ability to think about future investigation questions.'}, {'id': 3, 'title': 'Q3 — When should the audit row be written?', 'situation': 'The claim is about to move from `scored` to `priority_review`. Which sequence gives the strongest audit guarantee?', 'options': ['A. Move the claim first, then write the audit record later.', 'B. Let an external monitoring service notice the change and create the audit record.', 'C. Write the audit record in the same application code path before the result is released / the case moves on.', 'D. Write it once per hour in a batch.'], 'reference_reasoning': "Best reasoning: **C**. The lesson says the record is written in the Function's code path before the case moves on. A later observer creates an ordering gap: the business action can happen before the evidence exists.", 'correct_option': 3, 'fde_lens': 'Tests ordering, race-condition thinking, and evidence timing.'}, {'id': 4, 'title': 'Q4 — What should happen if the audit write is retried?', 'situation': 'The application writes an audit record successfully, but the request times out before the caller knows it succeeded. The operation is retried.', 'options': ['A. Generate a random audit ID each time so every attempt is visible.', 'B. Use a deterministic audit ID for the same event so a retry does not create a second logical audit row.', 'C. Ignore retries because audit writes should never fail.', 'D. Delete the first record if a duplicate is detected.'], 'reference_reasoning': 'Best reasoning: **B**. The lesson gives a deterministic audit_id so the same event has the same ID. This makes retries safe and supports append-only history without deleting the original row.', 'correct_option': 2, 'fde_lens': 'Tests idempotency and failure-mode thinking.'}, {'id': 5, 'title': 'Q5 — The audit Function is down for two hours', 'situation': 'Claims continue processing while the downstream Audit Function is unavailable. What is the correct interpretation?', 'options': ['A. The audit trail is definitely lost because the Audit Function was down.', 'B. Nothing can be known until the Function restarts.', 'C. The in-line application audit records should still exist; the downstream Function should resume from its Change Feed position and carry the backlog, provided the required changes are still available.', 'D. Azure Activity Log will reconstruct the missing case transitions.'], 'reference_reasoning': "Best reasoning: **C**. The lesson architecture makes the application's in-line write the recorder. The downstream Change Feed/Audit Function is transport/enrichment. Recovery must be checked by confirming the processor resumes and its position advances. The lesson does not establish that Activity Log can reconstruct application case transitions.", 'correct_option': 3, 'fde_lens': 'Tests resilience thinking and separation of source-of-truth from downstream processing.'}, {'id': 6, 'title': 'Q6 — The outage lasts two weeks', 'situation': 'The downstream processor has been broken for two weeks and relevant Change Feed history is no longer available. What should an FDE recommend?', 'options': ['A. Assume the Change Feed will always contain everything needed.', 'B. Restart the Function and report success without reconciliation.', 'C. Detect and alert on processing lag/retention risk, investigate the gap, and use the authoritative in-line audit records or another approved recovery source to reconcile what is missing from downstream storage.', 'D. Increase the retention period only after the incident has happened; that will recover old events.'], 'reference_reasoning': "Best reasoning: **C**. A mature FDE design does not rely on discovering a retention gap after it is unrecoverable. Monitoring should identify lag and risk early. The lesson's core architecture means the in-line audit records are the recorder; downstream archive processing must be reconciled if it falls behind. Do not claim a retention change can recreate events that have already aged out.", 'correct_option': 3, 'fde_lens': 'Tests operational monitoring, retention-risk thinking, and recovery planning.'}, {'id': 7, 'title': 'Q7 — What belongs in immutable storage?', 'situation': "The team proposes storing the claimant's full name, address and bank account in the WORM audit archive because “more information is always better for audits.”", 'options': ['A. Accept all PII; WORM means it is automatically compliant.', 'B. Keep identifiable personal data out where possible; use case/customer references and keep deletable identifying details in a normal store.', 'C. Put every field in WORM and ask the DPO to delete it later.', 'D. Never store any audit information at all because GDPR exists.'], 'reference_reasoning': 'Best reasoning: **B**. The lesson calls this the simplest approach: keep personal data out of immutable storage where possible and retain references instead. If data truly must be retained, legal basis and/or crypto-shredding may be considered with the appropriate owners. The FDE should not make the legal decision alone.', 'correct_option': 2, 'fde_lens': 'Tests data minimisation, compliance trade-offs, and knowing when to escalate to the data-protection/legal owner.'}, {'id': 8, 'title': 'Q8 — The WORM policy', 'situation': 'You are preparing production immutable storage. Which approach is safest?', 'options': ['A. Lock seven years immediately so nobody can change anything.', 'B. Start with an unlocked short-retention policy for testing; agree the retention period and permitted fields with the appropriate owner; lock only after that decision is approved.', 'C. Lock one day in production and increase it later if auditors complain.', 'D. Keep the policy permanently unlocked so engineers can fix mistakes.'], 'reference_reasoning': 'Best reasoning: **B**. Once locked, the lesson says retention can only be increased, not shortened or removed. The lab guidance is to test unlocked with short retention and lock only after the retention period has been agreed in writing.', 'correct_option': 2, 'fde_lens': 'Tests irreversible-decision thinking and governance.'}, {'id': 9, 'title': 'Q9 — How do you prove immutability?', 'situation': 'A project manager says: “Blob Storage is WORM, so just put that sentence in the architecture document.” What is the better FDE response?', 'options': ['A. Agree; documentation is enough.', 'B. Take a screenshot of the portal showing the WORM setting and stop there.', 'C. Perform evidence tests: write a record, attempt delete, attempt overwrite, attempt to shorten locked retention, read back and compare, and test search by case_id; preserve the refused-operation evidence.', 'D. Ask the cloud provider to certify it verbally.'], 'reference_reasoning': 'Best reasoning: **C**. The lesson explicitly says saying storage is immutable is a claim; demonstrating refused delete/overwrite/retention changes is evidence. The evidence should be retained for the governance appendix.', 'correct_option': 3, 'fde_lens': 'Tests evidence-driven engineering rather than assertion-driven engineering.'}, {'id': 10, 'title': 'Q10 — Activity Log vs application audit', 'situation': 'An auditor asks: “Who deleted the storage container?” Which source should you use, and what can it prove?', 'options': ['A. The application audit trail; it records all Azure infrastructure actions.', 'B. Azure Activity Log; it records Azure resource/control-plane actions. It complements the application audit trail but does not explain why a claim was flagged.', 'C. Cosmos Change Feed; it records every Azure subscription action.', 'D. Blob WORM; it automatically records who touched Azure resources.'], 'reference_reasoning': 'Best reasoning: **B**. The lesson says the application audit answers “what happened to this case?” while Azure Activity Log answers “who changed the infrastructure?” Both are needed for different questions. Activity Log also has limited default retention and may need export for longer regulatory needs.', 'correct_option': 2, 'fde_lens': 'Tests architectural boundary thinking and avoiding tool misuse.'}, {'id': 11, 'title': 'Q11 — Archive layout', 'situation': 'Two years later, an auditor asks for exactly one claim. You have about half a million audit rows in Blob Storage. Which layout best matches the question “show me this case”?', 'options': ['A. Put all records into large sequential batch files and scan every file.', 'B. Use a layout organised around case_id, such as `audit/case_id=<id>/<date>.json`, so the relevant case can be located directly.', 'C. Use a random folder name so access is unpredictable.', 'D. Store only a database backup because Blob search is unnecessary.'], 'reference_reasoning': 'Best reasoning: **B**. The lesson compares the folder path to a partition key: choose the layout around the question that must be cheap to answer. For this audit use case, that is usually finding one case.', 'correct_option': 2, 'fde_lens': 'Tests query-driven data layout and long-term operational thinking.'}, {'id': 12, 'title': 'Q12 — What does “audit trail complete” mean?', 'situation': 'Your dashboard says the audit pipeline is green. An auditor asks: “Can you prove every important transition was captured?” What is the strongest response?', 'options': ['A. “The Function had no errors, so yes.”', 'B. “The Change Feed is enabled, so yes.”', 'C. “Here is what our application records, here is the evidence that downstream delivery and storage are working, here is what the trail does not capture, and here is our recovery/monitoring approach.”', 'D. “Azure Activity Log confirms the application was healthy.”'], 'reference_reasoning': 'Best reasoning: **C**. FDE thinking distinguishes system health from evidence completeness. A strong answer states what is recorded, what is not, how downstream delivery is verified, how immutability is demonstrated, and what happens during outages.', 'correct_option': 3, 'fde_lens': 'Tests the overall “Can you prove it?” mindset.'}]

# Store your answers here.
# Example structure:
# ANSWERS = {
#   1: {"choice": 2, "reasoning": "I chose 2 because ...", "risk": "...", "evidence": "...", "owner": "..."},
# }

ANSWERS = {}

print("Loaded", len(QUESTIONS), "decision points.")


Loaded 12 decision points.


## Enter your answers

Run the next cell. It will guide you through each decision and collect your answer without requiring you to edit a large dictionary manually.


In [8]:
def collect_answers():
    answers = {}
    for q in QUESTIONS:
        print("\n" + "="*90)
        print(q["title"])
        print(q["situation"])
        for idx, option in enumerate(q["options"], 1):
            print(f"  {idx}. {option}")
        while True:
            try:
                choice = int(input("Your selected option (1-4): ").strip())
                if choice in (1,2,3,4):
                    break
            except ValueError:
                pass
            print("Please enter 1, 2, 3 or 4.")

        reasoning = input("Why did you choose it? ")
        risk = input("What could go wrong if your choice is wrong? ")
        evidence = input("What evidence would you ask for? ")
        owner = input("Who should approve/own the decision, if applicable? ")

        answers[q["id"]] = {
            "choice": choice,
            "reasoning": reasoning,
            "risk": risk,
            "evidence": evidence,
            "owner": owner
        }
    return answers

# Uncomment to start the exercise:
#ANSWERS = collect_answers()


### Optional: enter answers directly

If you already completed the exercise elsewhere, populate `ANSWERS` in the next cell. Keep the same structure shown below.


In [12]:
ANSWERS = {
  "1": {
    "choice": 2,
    "reasoning": "This ensures the audit record is captured independent of any changes to the Cosmos DB, and if the Change Feed is unavailable, it will still be available as a reference. Additionally, retention and WORM policies can be applied to the audit log.",
    "risk": "Risk of audit issues and loss of evidence.",
    "evidence": "The policy history, and proof that no changes were made to the audit log after writing.",
    "owner": "Head of Claims"
  },
  "2": {
    "choice": 3,
    "reasoning": "This provides all the information needed to reconstruct the record and evidence of changes of state, the actors, and the versions of models and prompts.",
    "risk": "Excessive storage; missing evidence.",
    "evidence": "The information needed for the audit record.",
    "owner": "Head of Claims"
  },
  "3": {
    "choice": 3,
    "reasoning": "Ensures consistency and transaction accuracy. Aligns with the state change.",
    "risk": "Risk of loss; audit issues.",
    "evidence": "Testing logs.",
    "owner": "Application Owner"
  },
  "4": {
    "choice": 2,
    "reasoning": "This serves as an idempotency key, so when the caller retries the timed-out request, the system recognizes the deterministic ID, realizes the work has already been completed successfully, and safely returns a success response without creating a duplicate, misleading log entry.",
    "risk": "If the choice was wrong, we would end up with duplicate audit entries for the same event.",
    "evidence": "Retry test logs showing a single audit row per event ID",
    "owner": "Technology leader"
  },
  "5": {
    "choice": 3,
    "reasoning": "If a downstream Audit Function goes down, the core application continues to process claims and write changes to the primary database. The Change Feed inherently preserves the sequential order of these records. Once the downstream Function restarts, it will read from its last checkpoint/pointer and process the backlog. The audit trail is not lost as long as the data retention period (time-to-live) of the source change feed accommodates the two-hour downtime.",
    "risk": "If Option C is incorrect due to systemic misconfiguration: (1) Data loss via retention expiration - if the Change Feed or event log has a retention window shorter than two hours, older unprocessed messages will expire and be permanently lost before the Function recovers. (2) Lack of local backlogs - if the upstream application relies on synchronous direct calls to the Audit Function rather than an asynchronous Change Feed pattern, claims might process without generating any audit records at all, making Option A true. (3) Checkpoint corruption - if the Function's pointer or checkpoint mechanism fails upon crash, it might skip records or process duplicates, causing data gaps or integrity issues.",
    "evidence": "Architecture diagram confirming that an asynchronous pattern (Change Feed, Service Bus, or Event Hub) sits between the claims processor and the Audit Function; retention policy settings showing the configured retention period or TTL for the source Change Feed or messaging queue exceeds two hours; application logs/telemetry showing the upstream claims processor is committing transactions without exceptions related to the downstream failure.",
    "owner": "Lead Cloud/Solutions Architect and Information Security Officer (ISO), jointly. The Architect confirms the technical recovery mechanism; the ISO signs off on whether the temporary state poses a compliance or data integrity risk during the catch-up period."
  },
  "6": {
    "choice": 3,
    "reasoning": "Ensures no loss of events and addresses processing lag.",
    "risk": "If this is wrongly configured, there would be significant loss of audit events and risk of lost transactions, unauthorized changes, etc.",
    "evidence": "Details on the outage, logs, evidence that the Function was not modified, and retention policies.",
    "owner": "Solution Architect"
  },
  "7": {
    "choice": 2,
    "reasoning": "Reduces exposure risk and ensures the ability to reconstruct the record when approved by a lawyer.",
    "risk": "PII exposure and leakage.",
    "evidence": "Evidence that no information has been stored or accessed outside the secure customer PII store.",
    "owner": "Head of Privacy"
  },
  "8": {
    "choice": 2,
    "reasoning": "Enables testing and verification of the process and aligns with the retention policy once approved.",
    "risk": "The record is locked and cannot be modified until the policy period expires.",
    "evidence": "Retention policy; test results for the WORM activities while the policy is unlocked.",
    "owner": "Head of Privacy"
  },
  "9": {
    "choice": 3,
    "reasoning": "Ensures reliable testing and verification through evidence.",
    "risk": "Risk of loss of audit records; lack of evidence to demonstrate a working WORM process.",
    "evidence": "Test results.",
    "owner": "Solution Architect"
  },
  "10": {
    "choice": 2,
    "reasoning": "Control-plane tracking: Azure Activity Log records management operations like container or resource deletions performed via Azure Resource Manager. Identity capture: it identifies the specific user or service principal (caller) who initiated the deletion. Default availability: it is captured by the platform automatically without requiring prior custom application configuration.",
    "risk": "Data gaps; retention policy violation; evidence gaps.",
    "evidence": "Activity Logs; access management policy.",
    "owner": "CISO and Cloud Infra Team"
  },
  "11": {
    "choice": 2,
    "reasoning": "Ease of access and low cost - reading only one file out of thousands.",
    "risk": "Slow search, higher costs, missed deadlines.",
    "evidence": "Storage layout proof: a sample path showing how folders use the case_id. Access test results: proof of how fast a single test query runs. Cost estimates: reports on expected read operations for audits.",
    "owner": "Data Architect and Compliance Lead"
  },
  "12": {
    "choice": 3,
    "reasoning": "Completeness: it addresses the exact question of proof across the entire pipeline, not just one component. Honesty: it openly states what the audit trail does not capture, which builds trust with the auditor. Accountability: it shows we actively monitor and recover the system if something fails. Rejection of false security: Options A, B, and D assume that a healthy Function, an enabled log feature, or a green status light equals verified, complete data capture, which is rarely true in a strict audit.",
    "risk": "Audit failure, data loss, loss of trust.",
    "evidence": "Trace logs, reconciliation reports, dead-letter queue logs, access and retention metrics.",
    "owner": "Compliance Officer, Architecture Lead, and Security Lead"
  }
}

In [20]:
def build_evaluation_prompt(question, trainee):
    return f'''
You are evaluating a Forward Deployed Engineer (FDE) trainee on an immutable audit-trail architecture.

IMPORTANT:
- Ground your assessment in the lesson reference below.
- Do not reward an answer merely because it uses sophisticated cloud terminology.
- Distinguish a technically plausible answer from the answer that best satisfies the scenario and lesson.
- Explain disagreements rather than simply saying correct/incorrect.
- Look for FDE behaviour: assumptions, failure modes, evidence, operational checks, ownership, governance and customer impact.

QUESTION:
{question["title"]}

SITUATION:
{question["situation"]}

OPTIONS:
{json.dumps(question["options"], indent=2)}

LESSON-GROUNDED REFERENCE:
{question["reference_reasoning"]}

TRAINEE ANSWER:
{json.dumps(trainee, indent=2)}

Return ONLY valid JSON with this shape:
{{
  "verdict": "Correct" | "Mostly correct" | "Needs improvement" | "Incorrect",
  "score_0_to_100": number,
  "selected_option": number,
  "expected_option": number,
  "why_agrees_or_differs": "clear explanation",
  "strong_points": ["..."],
  "missing_fde_thinking": ["..."],
  "risk_or_failure_mode": ["..."],
  "evidence_to_collect": ["..."],
  "recommended_answer": "plain-English recommended answer",
  "coaching_question": "one question that would make the trainee think deeper"
}}
'''
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY_1")
def call_gemini(prompt):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent"
    params = {"key": GEMINI_API_KEY}
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {
            "temperature": 0.2,
            "responseMimeType": "application/json"
        }
    }
    r = requests.post(url, params=params, json=payload, timeout=90)
    r.raise_for_status()
    data = r.json()
    return data["candidates"][0]["content"]["parts"][0]["text"]

def call_azure(prompt):
    if not AZURE_ENDPOINT or not AZURE_DEPLOYMENT:
        raise ValueError("Set AZURE_ENDPOINT and AZURE_DEPLOYMENT before using Azure.")
    url = (
        AZURE_ENDPOINT.rstrip("/")
        + f"/openai/deployments/{AZURE_DEPLOYMENT}/chat/completions"
        + f"?api-version={AZURE_API_VERSION}"
    )
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_API_KEY
    }
    payload = {
        "messages": [
            {"role": "system", "content": "You are a rigorous FDE architecture coach. Return valid JSON only."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.2,
        "response_format": {"type": "json_object"}
    }
    r = requests.post(url, headers=headers, json=payload, timeout=90)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

def evaluate_one(question, trainee):
    prompt = build_evaluation_prompt(question, trainee)
    raw = call_gemini(prompt) if PROVIDER == "gemini" else call_azure(prompt)
    return json.loads(raw)


## Run the evaluation

For best results, evaluate **after completing all decisions**. You can also evaluate selected questions by changing `QUESTION_IDS`.


In [21]:
QUESTION_IDS = sorted(ANSWERS.keys())

results = []

for qid_str in QUESTION_IDS:
    qid = int(qid_str) # Convert qid to an integer for comparison
    q = next(q for q in QUESTIONS if q["id"] == qid)
    trainee = ANSWERS[qid_str]

    try:
        result = evaluate_one(q, trainee)
        result["question_id"] = qid
        result["question_title"] = q["title"]
        results.append(result)

        print("\n" + "="*90)
        print(q["title"])
        print("Verdict:", result.get("verdict"))
        print("Score:", result.get("score_0_to_100"))
        print("Why:", result.get("why_agrees_or_differs"))
        print("Recommended:", result.get("recommended_answer"))
        print("Coaching question:", result.get("coaching_question"))
    except Exception as e:
        print(f"Evaluation failed for Q{qid}: {e}")

print(f"\nCompleted evaluation for {len(results)} question(s).")


Q1 — Who is the recorder?
Verdict: Incorrect
Score: 25
Why: The trainee selected option C (Ask Azure Activity Log to capture the case transition automatically), which is incorrect. The lesson explicitly states that the application code (Option B) should be the recorder for application-level state transitions, with Change Feed acting as a delivery mechanism. Azure Activity Log primarily captures control plane operations (resource management, e.g., who created a Cosmos DB account) rather than application-specific data plane events like a 'case transition' within a claims system. While the trainee's reasoning mentions good principles like independence, availability, retention, and WORM, these are general characteristics of a robust audit system and do not justify using Activity Log for application-level events, nor do they address the core problem of ensuring *all* rapid intermediate states are captured directly by the application.
Recommended: The application function should explicitly 

# FDE Review: What should you learn from the disagreements?

Do not stop at the score.

For every disagreement, ask:

### 1. Was my architecture technically wrong?
Or was it technically possible but weak against the **customer's actual requirement**?

### 2. What assumption did I make?
Examples:
- “Change Feed will capture every intermediate state.”
- “No Function error means no audit gap.”
- “Immutable means GDPR-compliant.”
- “Activity Log tells me what the application did.”
- “An archive is useful even if it is difficult to search.”

### 3. What evidence would settle the disagreement?
A good FDE does not argue from opinion when a small experiment, log, metric, or controlled failure test can settle the question.

### 4. Who owns the decision?
Some choices are engineering decisions. Others need the application owner, security team, data-protection/privacy owner, compliance, or the customer.

### 5. What would I change in production?
Write one concrete action for each weak answer.


In [22]:
# Produce a simple trainee summary.
if results:
    scores = [r["score_0_to_100"] for r in results if isinstance(r.get("score_0_to_100"), (int, float))]
    if scores:
        print(f"Average AI evaluation score: {sum(scores)/len(scores):.1f}/100")

    print("\nKey coaching points:")
    for r in results:
        missing = r.get("missing_fde_thinking", [])
        if missing:
            print(f"\nQ{r['question_id']} — {r['question_title']}")
            for item in missing:
                print(" •", item)
else:
    print("No results yet. Complete ANSWERS and run the evaluation cell.")


Average AI evaluation score: 76.3/100

Key coaching points:

Q1 — Q1 — Who is the recorder?
 • **Fundamental misunderstanding of 'recorder' vs. 'delivery mechanism':** The core lesson point. The trainee failed to identify the application code as the authoritative source for recording its own state changes, instead pointing to an external control plane logging service.
 • **Lack of distinction between control plane and data plane logging:** Azure Activity Log is designed for control plane events; application-level audit trails require data plane logging, which is typically instrumented within the application itself.
 • **Failure to address the 'rapid intermediate states' problem:** The trainee's reasoning does not explain how their chosen option (C) would prevent the loss of intermediate states, which is a key concern when considering Change Feed as the *recorder*.
 • **Assumptions about service capabilities:** Assumes Azure Activity Log can automatically capture granular application-sp

# Facilitator / FDE Challenge Round

After the AI evaluation, discuss these without looking at the reference:

1. **If the in-line audit write fails, should the business transaction fail too? Why?**
2. **If the audit write succeeds but the case update fails, what does your audit record now mean?**
3. **If the audit record contains a model version but not the prompt version, what future question might remain unanswered?**
4. **If WORM prevents deletion, what mistake becomes expensive the moment the policy is locked?**
5. **If the archive is immutable but cannot efficiently answer “show me this case,” is it a good audit trail?**
6. **What does “audit pipeline healthy” actually prove — and what does it not prove?**
7. **If the downstream Audit Function is down for a day, what metric/alert tells you whether you are approaching a recovery boundary?**
8. **Which parts of this architecture are technical decisions, and which require customer/legal/data-protection approval?**

### Final FDE statement

Complete this sentence:

> **“I would not tell the customer the audit trail is reliable until I can prove The audit trail has not been manipulated, the policies and evidence have been reviewed and tests executed, because evidence beats claims.”**


## Lesson source

This lab is grounded in the uploaded **Birlasoft FORGE FDE Academy — Sprint 2 Day 33: Immutable Audit Trail** material.

The lesson's core message is that the application code writes the audit record; the Change Feed carries those audit records downstream; Blob WORM provides evidence that cannot be deleted/overwritten once locked; and the design must explicitly account for PII, retention, searchability, and what the trail does not capture. fileciteturn0file0L36-L52

The source also explicitly frames the two-hour/two-week audit Function failure scenario and asks trainees to reason about restart/checkpointing and retention risk. fileciteturn0file0L265-L273
